# $$Attention-Mechanisms$$

`Attention mechanisms` are a core component of modern Large Language Models (LLMs). <br/>
They enable the model to dynamically focus on the **most relevant parts** of the input sequence when processing or generating text.

### We will implement four different variants of attention mechanisms
1. **Self-Attention** <br/>
Self-attention allows each token in a sequence to:
- Attend to all other tokens.
- Assign different importance (weights) to them.
- Produce a new representation based on a weighted combination of the entire sequence.

This enables the model to capture long-range dependencies in text.

2. **Causal Attention**<br/>
For autoregressive language models:
- Tokens must not attend to future tokens.
- Information flow is restricted to past and current positions.

Causal attention enforces this constraint, allowing the model to generate text **one token at a time.**

3.  **Attention Dropout**<br/>
To improve generalization:
- Attention weights are randomly masked using `dropout`.
- This prevents the model from over-relying on specific attention paths.
- Helps reduce overfitting during training.
  
4. **Multi-Head Attention**<br/>
Instead of using a single attention mechanism:
- Multiple attention heads operate in parallel.
- Each head learns to focus on different aspects of the input.
- Outputs are concatenated and projected to form the final representation.

Multi-head attention provides a richer and more expressive understanding of sequences.

## Problem with Long Sequences
Early neural architectures struggled to model long sequences effectively. <br/>
In tasks like language translation, word-by-word translation fails because:
- Languages have different grammatical structures
- Correct translation requires reordering words
- The meaning of a word often depends on distant context

### Encoder–Decoder Architecture
To handle sequence-to-sequence tasks, models commonly used an **encoder–decoder** structure:
> The **encoder** reads and processes the entire input sentence <br/>
> The **decoder** generates the output sentence one token at a time

### RNN-Based Encoder–Decoder Models
Before transformers, **Recurrent Neural Networks (RNNs)** were the dominant choice for encoder–decoder models.

Key characteristics:
- RNNs process input sequentially
- Each step depends on the previous hidden state
- They are naturally suited for ordered data like text

## $Hidden$ $State$ as a Context Representation
In an encoder–decoder RNN:
- The encoder compresses the entire input sequence into a **single final hidden state**
- This hidden state acts as a **fixed-size embedding** summarizing the whole sentence
- The decoder relies entirely on this hidden state to generate the output

### Core Limitation (Context Bottleneck)
The major weakness of encoder–decoder RNNs is that:
- The decoder cannot directly access earlier encoder hidden states
- All information must be stored in a single vector
- Long-range dependencies are easily lost

This **context bottleneck** becomes severe for long or complex sentences.

The inability of encoder–decoder RNNs to preserve and access detailed contextual information motivated the development of **attention mechanisms**, which allow models to directly focus on relevant parts of the input sequence.

<br/>

A few years after attention was introduced for RNN-based models, researchers discovered that **RNNs are not required** to build effective neural networks for natural language processing.

This led to the development of the **Transformer architecture**, which relies entirely on `attention mechanisms`.

## Self-Attention Mechanism
Self-attention allows each position in an input sequence to:
- **Attend to all other positions in the same sequence**
- **Compute its representation based on global context**
- **Capture both local and long-range dependencies efficiently**

## Attending to Different Parts of the Input with Self-Attention
**Role of Self-Attention**

Self-attention is the core building block of transformer-based Large Language Models.
It enables a model to compute representations of tokens by considering their relationships with all other tokens in the same input sequence.

## Meaning of “Self” in Self-Attention
The term **self** refers to the fact that attention is computed **within a single sequence.**

Self-attention:
- Relates different positions of the same input
- Learns dependencies among elements of the sequence
- Captures contextual relationships dynamically

<div style="text-align: center; margin-top: 20px;">
  <img 
    src="https://raw.githubusercontent.com/salavii/llm-from-scratch/main/images/context vector.png"
    style="width: 750px; border-radius: 10px; display: block; margin-left: auto; margin-right: auto;"
  >

  <p style="font-size: 16px; color: #333; font-weight: bold; margin-top: 10px;">
   The goal of self-attention is to compute a context vector for each input element that combines information from all other input elements.
  </p>
</div>


## Self-Attention Inputs and Context Vectors
Figure represents an input sequence x with T elements: <br/>
In practice, this sequence corresponds to text (e.g., a sentence) that has already been converted into **token embeddings.**

Example sentence:

“Your journey starts with one step.”

Each token (e.g., “Your”, “journey”, ...) is represented by a **d-dimensional embedding vector.**
For illustration, the figure uses **3-dimensional embeddings** to keep the example compact.

**Goal of Self-Attention: Context Vectors**<br/>
- A context vector can be interpreted as an enriched embedding:
- It contains information about the token itself
- It also incorporates information from other tokens in the same sequence

### Why Context Vectors Matter in LLMs

Context vectors are crucial because they allow the model to build context-aware representations of tokens.
This helps LLMs understand relationships and relevance between words in a sentence.

Later, trainable weights will be introduced so the model can learn how to construct context vectors that are useful for next-token prediction

<br/>

In [2]:
import torch
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

#### Computing Attention Scores for a Query Token

In [3]:
# Select the query token (second token in the sequence)
query = inputs[1]

# Allocate a tensor to store attention scores
attn_scores_2 = torch.empty(inputs.shape[0])

# Compute dot-product attention scores
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
    
# Print raw (unnormalized) attention scores
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


## Dot Product as a Similarity Measure
Beyond being a mathematical operation, the dot product measures similarity between two vectors.
- A higher dot product indicates stronger alignment
- A lower dot product indicates weaker alignment

This makes the dot product useful for comparing embeddings

## Role in Self-Attention
In self-attention mechanisms:
- Dot products are used to compute attention scores
- These scores determine how strongly one token attends to another

Higher dot product values correspond to higher attention scores

# Normalizing Attention Scores
The attention scores computed in the previous step are **raw similarity values.** <BR/>
While they indicate how related tokens are, they are not directly interpretable.

To make attention meaningful and stable, these scores are normalized.

### Attention Weights
The goal of normalization is to obtain **attention weights that**:
- Are non-negative
- Sum up to 1

This allows attention to be interpreted as a **distribution of focus** over the input tokens

In [4]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


# Normalizing Attention Scores with Softmax
- Handles extreme values more effectively
- Ensures all attention weights are positive
- Provides smoother and more stable gradients during training

**Softmax turns raw attention scores into stable, interpretable attention weights.**

In [5]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)
attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


## Numerical Stability Considerations
A naive softmax implementation may suffer from numerical instability, such as:
- Overflow when input values are very large
- Underflow when input values are very small

These issues can lead to invalid values and unstable training behavior

### Practical Recommendation

In practice, it is recommended to use `PyTorch’s optimized` softmax implementation

In [6]:
attn_weights_2 = torch.softmax(attn_scores_2, dim= 0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [7]:
query = inputs[1]
# Initialize an empty context vector
context_vec_2 = torch.zeros(query.shape)
context_vec_2

tensor([0., 0., 0.])

In [8]:
# Compute the context vector as a weighted sum of input embeddings
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)


tensor([0.4419, 0.6515, 0.5683])


# Computing attention weights for all input tokens

<div style="text-align: center; margin-top: 20px;">
  <img 
    src="https://raw.githubusercontent.com/salavii/llm-from-scratch/main/images/attention.png"
    style="width: 750px; border-radius: 10px; display: block; margin-left: auto; margin-right: auto;"
  >


In [9]:
attn_scores = torch.empty(6, 6)
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


## Computing Attention Scores with Matrix Multiplication
Each element in the attention score tensor represents the similarity between a pair of input tokens.

Previously, these scores were computed using nested Python loops.
While conceptually clear, this approach is inefficient.

### Efficient Implementation
attn_scores = inputs @ inputs.T


In [10]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

<br/>

In step , we normalize each row so that the values in each row sum to 1:

In [11]:
attn_weights = torch.softmax(attn_scores, dim=1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

<br/>

In the third and final step, we use these attention weights to compute all
context vectors via matrix multiplication:

In [12]:
all_context_vec = attn_weights @ inputs
all_context_vec

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

We can double-check that the code is correct by comparing the second row with the
context vector z(2) that we computed 

In [13]:
print("Previous 2nd context vector:", context_vec_2)


Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


# Implementing self-attention with trainable weights

### Self-Attention with Trainable Weights (Overview)
Self-attention with trainable weights builds on the same core idea as the simple version:
we compute **context vectors** as **weighted sums** over the input vectors, where the
weights depend on a specific input token.

### Key Difference: Trainable Weight Matrices
The main difference is the introduction of **trainable weight** matrices that are updated during training.

These matrices allow the attention module to learn how to produce “good” context vectors that are useful for the model’s objective (e.g., next-token prediction)

### Step-by-Step Self-Attention with Trainable Weights
To implement self-attention with trainable parameters, we introduce three learnable weight matrices:
- QW (Query projection)
- KW (Key projection)
- VW(Value projection)

These matrices project each embedded input token x(i) into three different vectors:
- Query vector q(i)
- Key vector k(i)
- Value vector v(i)


### Why Q, K, and V?
Using separate trainable projections allows the attention module to learn:
- How to compare tokens (via queries and keys)
- What information to extract and combine (via values)

In [14]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2  # The output embedding size
x_2

tensor([0.5500, 0.8700, 0.6600])

In [15]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [16]:
query_2 = x_2 @ W_query 
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306, 1.4551])


The output for the query results in a two-dimensional vector since we set the number
of columns of the corresponding weight matrix, via d_out, to 2

- query_2 represents what the token is looking for
- key_2 represents what the token offers for matching
- value_2 contains the information that will be aggregated

All three vectors are different projections of the same input embedding

### Weight parameters (W)
→ transform input embeddings into Query (Q), Key (K), and Value (V) vectors

Query (Q) and Key (K)
→ are used to compute **attention scores.**

### Softmax
→ converts attention scores into **attention weights.**

### Attention weights × Value (V)
→ produces the **context vector**

**Weight parameters are learned and fixed.** <br/>
**Attention weights are computed and dynamic**

In [17]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


The second step is to compute the attention scores

First, let’s compute the attention score ω22:


In [18]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print (attn_score_22)

tensor(1.8524)


we can generalize this computation to all attention scores via matrix
multiplication:

In [19]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2


tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

We divide by √dₖ so that softmax does not become too sharp.

`d_k` is the dimension of the key vectors,
and `sqrt(d_k)` is used to scale attention scores

We compute the attention weights by scaling the attention scores and
using the softmax function. However, now we scale the attention scores by dividing
them by the square root of the embedding dimension of the keys (taking the square
root is mathematically the same as exponentiating by 0.5):

In [20]:
# This is used to scale the attention scores for numerical stability
d_k = keys.shape[-1]

# Scale the attention scores by sqrt(d_k) and apply softmax
# This converts raw attention scores into normalized attention weights
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


### Scaled Dot-Product Attention
The purpose of scaling attention scores by the square root of the embedding dimension is to improve training stability and avoid vanishing gradients.

In large language models such as GPT, embedding dimensions are often very large (e.g., greater than 1000). As the embedding dimension increases, dot products between queries and keys can grow large in magnitude

final step is to compute the context vectors

In [21]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3061, 0.8210])

So far, we’ve only computed a single context vector, z(2). Next, we will generalize the
code to compute all context vectors in the input sequence, z(1) to z(T).

## Implementing a Compact Self-Attention Class

So far, we have computed self-attention outputs step by step to clearly understand each individual operation.

While this approach is useful for learning and illustration, it is not practical for real-world model implementations. In practice—especially when building large language models—it is more convenient to organize the self-attention logic into a reusable Python class

In [22]:
import torch.nn as nn
class SelfAttention_V1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        # Learnable projection matrices
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        # Project input embeddings into Query, Key, and Value
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value

        # Compute attention scores
        attn_scores = queries @ keys.T

        # Scale attention scores and apply softmax to obtain attention weights
        attn_weigths = torch.softmax(
            attn_scores / keys.shape[-1] ** 0.5, dim=-1
        )

        # Compute context vectors as weighted sum of values
        context_vec = attn_weigths @ values
        return context_vec
        

In [23]:
torch.manual_seed(123)
sa_v1 = SelfAttention_V1(d_in, d_out)
print (sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


The core self-attention computation does not change in this step. The only modification is **how the trainable weight matrices Wq, Wk, and Wv** are implemented.

Instead of manually defining these matrices using `nn.Parameter(torch.rand(...))`, we can use PyTorch’s `nn.Linear` layers with the bias term disabled. Both approaches perform the same matrix multiplication, but `nn.Linear` provides optimized weight initialization and better integration with PyTorch’s training utilities.

Using `nn.Linear` results in cleaner code and more stable and effective model training, while preserving the exact same self-attention behavior

In [24]:
class SelfAttention_V2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weigths = (
            torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        )
        context_vec = attn_weights @ values
        return context_vec
    

In [25]:
torch.manual_seed(123)
sa_v2 = SelfAttention_V2(d_in, d_out)
print(sa_v2(inputs))

tensor([[-0.5471, -0.0925],
        [-0.5652, -0.1222],
        [-0.5649, -0.1220],
        [-0.5483, -0.1179],
        [-0.5475, -0.1142],
        [-0.5522, -0.1204]], grad_fn=<MmBackward0>)


# Causal and Multi-Head Attention
The self-attention mechanism can be further enhanced by introducing **causal** and **multi-head components**

`Causal attention` restricts the model from attending to future tokens, ensuring that each prediction depends only on past information. This is essential for language modeling tasks.

`Multi-head attention` splits the attention mechanism into multiple parallel heads. Each head learns different aspects of the input representations, allowing the model to attend to information from multiple representation subspaces simultaneously. This improves performance on complex tasks

## Causal (Masked) Attention
Causal attention is a specialized form of self-attention that prevents the model from attending to future tokens in a sequence. When predicting a token, the model is restricted to attend only to the current and previous tokens.

This behavior is essential for language modeling tasks, where future information must not be accessed. Causal attention is implemented by masking out attention weights corresponding to future positions (above the diagonal of the attention matrix) and then normalizing the remaining weights so that each row sums to one

## Applying a Causal Attention Mask
The first step is to compute the attention weights using the softmax function, as in standard self-attention. In subsequent steps, attention weights corresponding to future tokens are masked out, and the remaining weights are renormalized so that each row sums to one.

This masking process enforces the causal constraint required for language modeling tasks

<div style="text-align: center; margin-top: 20px;">
  <img 
    src="https://raw.githubusercontent.com/salavii/llm-from-scratch/main/images/MASK.png"
    style="width: 750px; border-radius: 10px; display: block; margin-left: auto; margin-right: auto;"
  >


In [26]:
queries = sa_v2.W_query(inputs)    
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


We can implement the second step using PyTorch’s tril function to create a mask
where the values above the diagonal are zero:

In [27]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


Now, we can multiply this mask with the attention weights to zero-out the values above
the diagonal:


In [28]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


The third step is to renormalize the attention weights to sum up to 1 again in each
row. We can achieve this by dividing each element in each row by the sum in each row:

In [29]:
row_sum = masked_simple.sum(dim=-1, keepdim=True)  # keepdim -> The shape of the matrix is preserved.
masked_simple_norm = masked_simple / row_sum
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


## Efficient Causal Masking with Softmax
The softmax function converts input scores into a probability distribution. When negative infinity (−∞) values are present, softmax assigns them zero probability because 
e−∞→0
e
−∞
→0.
By setting attention scores of masked (future) positions to −∞ before applying softmax, those positions automatically receive zero weight. This produces properly normalized masked attention weights in a single step, eliminating the need for an additional renormalization step

In [30]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


In [31]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


## Next Step: Reducing Overfitting
At this point, the masked attention weights can be used to compute the context vectors by multiplying them with the value vectors. However, before doing so, we introduce a small additional modification to the causal attention mechanism. This tweak helps reduce overfitting during LLM training and improves generalization

## Dropout in Attention Mechanisms
Dropout is a regularization technique that randomly masks elements during training to reduce overfitting. In transformer-based models, dropout is commonly applied to the attention mechanism, typically after computing the attention weights. This prevents the model from relying too heavily on specific attention paths. Dropout is only active during training and is disabled during inference

In [32]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [33]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


<br/>


Before building the class, we ensure the implementation supports batched inputs, since training uses batches produced by the data loader. Therefore, the attention module should handle input tensors shaped like (B, T, d_in) rather than a single sequence (T, d_in). For simplicity, we simulate batched data by duplicating the same example sequence

In [34]:
# Two inputs with six tokens each; each token has embedding dimension 3.
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [35]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        #The register_buffer call is also a new addition (more information is provided in the following text).
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
        attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        
        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ values
        return context_vec

`register_buffer` in Causal Attention

The register_buffer method is used to store tensors that are not trainable parameters but should remain part of the model. In this case, the causal attention mask is registered as a buffer. This ensures that the mask is automatically moved to the correct device (CPU or GPU) along with the model, preventing device mismatch errors during training and inference

In [36]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


The resulting context vector is a three-dimensional tensor where each token is now
represented by a two-dimensional embedding

# Extending single-head attention to multi-head attention
Multi-head attention extends the idea of single-head (causal) attention by running **multiple attention mechanisms in parallel**, called heads. Each head processes the same input sequence independently but learns different attention patterns.

A single causal attention module can be seen as **single-head attention**, where only one set of attention weights is used to compute relationships between tokens.

Using multiple attention heads allows the model to:
- Capture different types of dependencies simultaneously
- Attend to different positions or features of the sequence
- Learn richer and more expressive representations

## Stacking Multiple Single-Head Attention Layers
**Basic Idea**
One intuitive way to implement multi-head attention is to **stack multiple single-head self-attention modules**. Each module has its own set of parameters and processes the same input sequence independently.

### How It Works
Each head:
- Receives the same input embeddings
- Uses its own attention weights
- The outputs of all heads are:
  - Combined (e.g., concatenated)
  - Used as the final multi-head attention output

### Core Idea
The main idea behind multi-head attention is to run the attention mechanism **multiple times in parallel**, each time using a **different learned linear projection** of the input.

These linear projections are obtained by multiplying the same input embeddings with different weight matrices, producing distinct query, key, and value representations for each attention head

In [37]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]            
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [38]:
torch.manual_seed(123)
context_length = batch.shape[1] # This is the number of tokens
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(
    d_in, d_out, context_length, 0.0, num_heads=2
)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


## Implementing multi-head attention with weight splits
So far, multi-head attention was implemented using a `MultiHeadAttentionWrapper` that stacked multiple independent single-head attention modules. Each attention head was represented by a separate `CausalAttention` instance, and their outputs were concatenated.

While this approach is conceptually simple, it is not computationally efficient.

### Integrating Multi-Head Attention into a Single Module
Instead of maintaining separate `CausalAttention` modules for each head, we can integrate the multi-head mechanism into a single `MultiHeadAttention` class.

In this optimized implementation:
- Queries, keys, and values are computed using single large projection matrices
- The projected tensors are reshaped to split the embedding dimension into multiple heads
- Attention is computed in parallel across all heads
- The outputs of the heads are then combined into a single context representation

In [41]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads  ## Size of the subspace each attention head operates on

        self.W_query = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias= qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias= qkv_bias) 
        self.out_proj = nn.Linear(d_out, d_out)  # This layer mixes information across different attention heads
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
                             "mask",
                              torch.triu(torch.ones(context_length, context_length),
                              diagonal=1)
                            )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)      
        queries = self.W_query(x)   
        values = self.W_value(x)

        
        # Divide the embedding dimension across attention heads
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)      
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)  
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)         
        queries = queries.transpose(1, 2)   
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)  
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]   
        attn_scores.masked_fill_(mask_bool, -torch.inf)    
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)  
        context_vec = context_vec.contiguous().view(
                                b, num_tokens, self.d_out
                                                    )
        context_vec = self.out_proj(context_vec)   
        return context_vec

### Transposing Tensors for Batched Attention Computation

After reshaping, the query, key, and value tensors have shape
`(batch_size, num_tokens, num_heads, head_dim).`

These tensors are then transposed to move the num_heads dimension before the num_tokens dimension, resulting in the shape

In [42]:
torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


Practical Notes on Multi-Head Attention Dimensions

We have now implemented the MultiHeadAttention class that will be used when building and training the language model. While the implementation is fully functional, relatively small embedding dimensions and numbers of attention heads were chosen to keep the outputs readable and easy to inspect.

In practice, large language models use much larger dimensions. For example, the smallest GPT-2 model (117 million parameters) uses 12 attention heads with a context embedding size of 768, while the largest GPT-2 model (1.5 billion parameters) uses 25 attention heads with a context embedding size of 1600.

In GPT models, the dimensionality of the token embeddings and the context embeddings are the same, meaning that the input and output dimensions of the attention layer satisfy d_in = d_out. This design choice ensures compatibility with residual connections and subsequent Transformer layers